In [ ]:
# ==========================================
# MVP1 ADAPTED DIFFUSION PIPELINE (SINGLE FILE)
# CHANGED VERSION: MODEL FUTURE YIELD-CHANGES
# ==========================================
# Key Improvements:
# - Per-tenor normalization for LEVELS (conditioning input)
# - Per-tenor normalization for CHANGES (training target)
# - Time-based train/val/test split
# - Sinusoidal timestep embeddings
# - Validation loss tracking
# - Model checkpoint saving/loading
# - Flexible scenario generation from ANY input curve
# - Configurable noise schedule
# ==========================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ==========================================
# 1. DATASET WITH SEPARATE NORMALIZATION
#    LEVELS for conditioning
#    CHANGES for targets
# ==========================================
class YieldPathDataset(torch.utils.data.Dataset):
    def __init__(self, df, horizon=60, yield_cols=None,
                 mu_y=None, std_y=None, mu_dy=None, std_dy=None):
        self.horizon = horizon

        # Use provided yield columns for consistency across train/val/test
        self.yield_cols = yield_cols if yield_cols else [c for c in df.columns if 'Y_DGS' in c]

        # Raw yield levels
        y = df[self.yield_cols].values.astype(np.float32)

        # CHANGED: compute first differences (monthly yield changes)
        # dy[t] = y[t+1] - y[t]
        dy = y[1:] - y[:-1]

        # Normalize LEVELS for conditioning input
        if mu_y is None or std_y is None:
            self.mu_y = y.mean(axis=0)
            self.std_y = y.std(axis=0) + 1e-8
        else:
            self.mu_y = mu_y
            self.std_y = std_y

        # CHANGED: normalize CHANGES separately for prediction target
        if mu_dy is None or std_dy is None:
            self.mu_dy = dy.mean(axis=0)
            self.std_dy = dy.std(axis=0) + 1e-8
        else:
            self.mu_dy = mu_dy
            self.std_dy = std_dy

        # Normalized current level
        y_norm = (y - self.mu_y) / self.std_y

        # CHANGED: normalized changes
        dy_norm = (dy - self.mu_dy) / self.std_dy

        self.paths = []
        self.y_starts = []

        # CHANGED:
        # condition = normalized current level at time i
        # target    = normalized future changes from i->i+1 up to i+horizon-1->i+horizon
        for i in range(len(y) - horizon - 1):
            self.paths.append(dy_norm[i:i+horizon])   # future normalized changes
            self.y_starts.append(y_norm[i])           # normalized current yield level

        self.paths = torch.tensor(np.array(self.paths), dtype=torch.float32).unsqueeze(1)
        self.y_starts = torch.tensor(np.array(self.y_starts), dtype=torch.float32)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.paths[idx], self.y_starts[idx]


# ==========================================
# 2. SINUSOIDAL TIME EMBEDDING
# ==========================================
def sinusoidal_embedding(timesteps, dim):
    device = timesteps.device
    half_dim = dim // 2
    emb_factor = np.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=device) * -emb_factor)
    emb = timesteps.float().unsqueeze(1) * emb.unsqueeze(0)
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


# ==========================================
# 3. MODEL (RESIDUAL CNN DENOISER)
# ==========================================
class PathResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.net(x)


class PathDenoiser(nn.Module):
    def __init__(self, tenors=11, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.cond_mlp = nn.Sequential(
            nn.Linear(tenors, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.input_conv = nn.Conv2d(1, hidden_dim, 3, padding=1)
        self.res_layers = nn.ModuleList([PathResBlock(hidden_dim) for _ in range(4)])
        self.output_conv = nn.Conv2d(hidden_dim, 1, 3, padding=1)

    def forward(self, x, t, y_start):
        t_emb = sinusoidal_embedding(t, self.hidden_dim)
        t_emb = self.time_mlp(t_emb).view(-1, self.hidden_dim, 1, 1)

        c_emb = self.cond_mlp(y_start).view(-1, self.hidden_dim, 1, 1)

        h = self.input_conv(x) + t_emb + c_emb

        for layer in self.res_layers:
            h = layer(h)

        return self.output_conv(h)


# ==========================================
# 4. NOISE SCHEDULE (LINEAR + COSINE)
# ==========================================
def make_beta_schedule(T, schedule="cosine", device="cpu"):
    if schedule == "linear":
        return torch.linspace(1e-4, 0.02, T, device=device)
    elif schedule == "cosine":
        steps = T + 1
        x = torch.linspace(0, T, steps, device=device)
        alphas_cumprod = torch.cos(((x / T) + 0.008) / 1.008 * torch.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        beta = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clamp(beta, 1e-4, 0.999)
    else:
        raise ValueError("schedule must be 'linear' or 'cosine'")


# ==========================================
# 5. DIFFUSION ENGINE
# ==========================================
class PathDiffusionEngine:
    def __init__(self, model, horizon=60, tenors=11, T=400, device='cpu'):
        self.model = model
        self.horizon = horizon
        self.tenors = tenors
        self.T = T
        self.device = device

        self.beta = make_beta_schedule(T, "cosine", device)
        self.alpha = 1.0 - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def train_loss(self, path_0, y_start):
        # NOTE:
        # path_0 is now the clean future CHANGE path, not future levels
        batch = path_0.shape[0]
        t = torch.randint(0, self.T, (batch,), device=self.device)

        noise = torch.randn_like(path_0)

        a_hat = self.alpha_hat[t].view(-1, 1, 1, 1)
        path_t = torch.sqrt(a_hat) * path_0 + torch.sqrt(1 - a_hat) * noise

        # v-objective
        v_target = torch.sqrt(a_hat) * noise - torch.sqrt(1 - a_hat) * path_0
        v_pred = self.model(path_t, t, y_start)

        return F.mse_loss(v_pred, v_target)

    @torch.no_grad()
    def sample_path(self, y_start):
        # Output is now a generated future CHANGE path in normalized space
        n = y_start.shape[0]
        x = torch.randn((n, 1, self.horizon, self.tenors), device=self.device)

        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            v = self.model(x, t, y_start)

            a_hat = self.alpha_hat[i]
            x0 = torch.sqrt(a_hat) * x - torch.sqrt(1 - a_hat) * v

            if i > 0:
                a_prev = self.alpha_hat[i - 1]
                direction = torch.sqrt(1 - a_prev) * ((x - torch.sqrt(a_hat) * x0) / torch.sqrt(1 - a_hat))
                x = torch.sqrt(a_prev) * x0 + direction
            else:
                x = x0

        return x.squeeze(1)


# ==========================================
# 6. TRAIN / VALIDATION LOSS
# ==========================================
@torch.no_grad()
def evaluate_loss(engine, model, loader, device):
    model.eval()
    losses = []
    for p, ys in loader:
        p, ys = p.to(device), ys.to(device)
        loss = engine.train_loss(p, ys)
        losses.append(loss.item())
    return float(np.mean(losses))


# ==========================================
# 7. SCENARIO GENERATION FROM ANY CURVE
# ==========================================
def generate_scenarios(model, engine, start_curve,
                       mu_y, std_y, mu_dy, std_dy, yield_cols,
                       num_scenarios=200, batch_size=50, filename="scenarios.csv"):

    model.eval()
    device = engine.device

    # Current yield curve LEVEL (raw space)
    start_curve = np.array(start_curve, dtype=np.float32)

    # Normalize the current level for conditioning
    start_norm = (start_curve - mu_y) / std_y
    start_tensor = torch.tensor(start_norm).unsqueeze(0).to(device)

    all_paths = []
    batches = int(np.ceil(num_scenarios / batch_size))

    for b in range(batches):
        size = min(batch_size, num_scenarios - b * batch_size)
        y_batch = start_tensor.repeat(size, 1)

        # CHANGED: sample normalized future CHANGES
        paths_dy_norm = engine.sample_path(y_batch).cpu().numpy()

        # CHANGED: denormalize changes using CHANGE statistics
        paths_dy = paths_dy_norm * std_dy.reshape(1, 1, -1) + mu_dy.reshape(1, 1, -1)

        # CHANGED: reconstruct future LEVELS from cumulative sum of changes
        paths_levels = start_curve.reshape(1, 1, -1) + np.cumsum(paths_dy, axis=1)

        all_paths.append(paths_levels)

    all_paths = np.vstack(all_paths)

    rows = []
    for s in range(num_scenarios):
        for t in range(engine.horizon):
            row = {"Scenario_ID": s + 1, "Month": t + 1}
            for i, col in enumerate(yield_cols):
                row[col] = all_paths[s, t, i]
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    return df


# ==========================================
# 8. MAIN TRAINING PIPELINE
# ==========================================
if _name_ == "_main_":

    HORIZON = 60
    TENORS = 11
    EPOCHS = 25  #stop around 25

    df = pd.read_csv("raw_data.csv", index_col=0, parse_dates=True)

    # Time-based split
    df_train = df.loc[:'2017']
    df_val = df.loc['2018':'2020']
    df_test = df.loc['2021':]

    # Train dataset learns:
    # - level normalization for conditioning
    # - change normalization for targets
    train_ds = YieldPathDataset(df_train, HORIZON)

    # CHANGED: validation reuses BOTH level and change normalization from train
    val_ds = YieldPathDataset(
        df_val,
        HORIZON,
        yield_cols=train_ds.yield_cols,
        mu_y=train_ds.mu_y,
        std_y=train_ds.std_y,
        mu_dy=train_ds.mu_dy,
        std_dy=train_ds.std_dy
    )

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=32)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PathDenoiser(TENORS).to(device)
    engine = PathDiffusionEngine(model, HORIZON, TENORS, device=device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    train_losses, val_losses = [], []
    best_val = float("inf")

    print("Training...")

    for epoch in range(EPOCHS):
        model.train()
        batch_losses = []

        for p, ys in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            p, ys = p.to(device), ys.to(device)

            loss = engine.train_loss(p, ys)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)
        val_loss = evaluate_loss(engine, model, val_loader, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1} | Train: {train_loss:.5f} | Val: {val_loss:.5f}")

        # Save best model
        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                "model_state_dict": model.state_dict(),
                "mu_y": train_ds.mu_y,
                "std_y": train_ds.std_y,
                # CHANGED: save change normalization too
                "mu_dy": train_ds.mu_dy,
                "std_dy": train_ds.std_dy,
                "yield_cols": train_ds.yield_cols,
                "horizon": HORIZON,
                "tenors": TENORS,
                "T": engine.T
            }, "best_model.pt")

    # Plot loss curves
    plt.plot(train_losses, label="Train")
    plt.plot(val_losses, label="Val")
    plt.legend()
    plt.title("Loss curves")
    plt.show()

    # Example generation from latest observed curve
    sample_curve = df.iloc[-1][train_ds.yield_cols].values

    generate_scenarios(
        model, engine, sample_curve,
        train_ds.mu_y, train_ds.std_y,
        train_ds.mu_dy, train_ds.std_dy,
        train_ds.yield_cols,
        num_scenarios=200
    )    